In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "intfloat/multilingual-e5-small"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 512
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

{'model_name': 'intfloat/multilingual-e5-small', 'dataset': 'glue/stsb', 'split': 'validation', 'subset_size': 512, 'device': 'mps', 'batch_size': 128, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

subset_indices = np.random.RandomState(seed).choice(len(df), size=min(subset_size, len(df)), replace=False)
subset_indices = np.sort(subset_indices)
df = df.iloc[subset_indices].reset_index(drop=True)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 512, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0           People are playing cricket.   
1           A man is finding something.   
2             A man is playing a flute.   
3                      A man is crying.   
4  The lady cracked an egg into a bowl.   

                               sentence2  label  
0               Men are playing cricket.  3.200  
1          A woman is slicing something.  0.800  
2               A man is playing guitar.  2.167  
3                    A woman is dancing.  0.600  
4  The man is cracking eggs into a bowl.  2.600  


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

In [ ]:
def e5_format(texts, instruction="Retrieve semantically similar text"):
    prefix = f"query: {instruction} | "
    return [prefix + str(t).replace("\n", " ").strip() for t in texts]

sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

formatted1 = e5_format(sentences1)
formatted2 = e5_format(sentences2)

emb1 = model.encode(
    formatted1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    formatted2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

In [ ]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["formatted_sentence1"] = formatted1
results_df["formatted_sentence2"] = formatted2
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

pred_rank = results_df["predicted_score_0_5"].rank(method="average", ascending=False)
label_rank = results_df["label"].rank(method="average", ascending=False)
results_df["pred_rank"] = pred_rank.astype(float)
results_df["label_rank"] = label_rank.astype(float)
results_df["rank_diff"] = results_df["pred_rank"] - results_df["label_rank"]
results_df["abs_rank_diff"] = results_df["rank_diff"].abs()

rank_corr = pearsonr(results_df["pred_rank"], results_df["label_rank"]).statistic
top1_label_idx = int(results_df["label"].idxmax())
top1_pred_idx = int(results_df["predicted_score_0_5"].idxmax())
top1_overlap = int(top1_label_idx == top1_pred_idx)
top10_label = set(results_df.nlargest(min(10, len(results_df)), "label").index.tolist())
top10_pred = set(results_df.nlargest(min(10, len(results_df)), "predicted_score_0_5").index.tolist())
top10_jaccard = len(top10_label & top10_pred) / len(top10_label | top10_pred)

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))
mean_abs_rank_diff = float(results_df["abs_rank_diff"].mean())
median_abs_rank_diff = float(results_df["abs_rank_diff"].median())

print(results_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error", "pred_rank", "label_rank", "abs_rank_diff"]].head(10))

In [ ]:
top_k = 10

largest_rank_mismatches = results_df.nlargest(top_k, "abs_rank_diff")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity",
    "absolute_error", "pred_rank", "label_rank", "rank_diff", "abs_rank_diff"
]].reset_index(drop=True)

best_agreement = results_df.nsmallest(top_k, "abs_rank_diff")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity",
    "absolute_error", "pred_rank", "label_rank", "rank_diff", "abs_rank_diff"
]].reset_index(drop=True)

print("Largest rank mismatches:")
print(largest_rank_mismatches.to_string(index=False))
print()
print("Best rank agreement examples:")
print(best_agreement.to_string(index=False))

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"rank_pearson_correlation: {rank_corr:.6f}")
print(f"top1_rank_agreement: {top1_overlap}")
print(f"top10_jaccard_agreement: {top10_jaccard:.6f}")
print(f"mean_absolute_rank_difference: {mean_abs_rank_diff:.6f}")
print(f"median_absolute_rank_difference: {median_abs_rank_diff:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")